In [3]:
import pandas as pd
import numpy as np
import altair as alt
import requests as rqs

In [4]:
import ssl
from pathlib import Path
from requests.adapters import HTTPAdapter


class IBGETLSAdapter(HTTPAdapter):
    """Completa a cadeia do IBGE sem desativar a valida??o TLS."""

    def __init__(self, certificado_intermediario, **kwargs):
        self.ssl_context = ssl.create_default_context(cafile=rqs.certs.where())
        # O intermedi?rio deve encadear at? uma raiz confi?vel, sem cadeia parcial.
        self.ssl_context.verify_flags &= ~ssl.VERIFY_X509_PARTIAL_CHAIN
        self.ssl_context.load_verify_locations(cafile=str(certificado_intermediario))
        super().__init__(**kwargs)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        pool_kwargs['ssl_context'] = self.ssl_context
        return super().init_poolmanager(connections, maxsize, block=block, **pool_kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs['ssl_context'] = self.ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)

In [5]:
id_milho = 2711
id_soja = 2713
id_cana = 2696
#Área plantada Hectares
area_plant_H = 109
#Área plantada - percentual do total geral %
area_plant_100 = 1000109
#Área colhida Hectares
area_colhida_H = 216
#Área colhida - percentual do total geral %
area_colhida_100 = 1000216
#Quantidade produzida Toneladas
quant_prod_T = 214
#Rendimento médio da produção Quilogramas por Hectare
rend_medio_Kg_H = 112
#Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992],
#  Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], 
# Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
valor_prod_moeda = 215
#Valor da produção - percentual do total geral %
valor_prod_100 = 1000215
tabela_lavouras_temporárias = 1612

certificado_intermediario = Path("certs/SectigoPublicServerAuthenticationCADVR36.pem")

In [6]:
def requisicao_certificada(url: str, parametros: dict = {}) -> dict:
    with rqs.Session() as sessao:
        sessao.mount("https://servicodados.ibge.gov.br/", IBGETLSAdapter(certificado_intermediario))
        response = sessao.get(url, params=parametros, timeout=30)
        response.raise_for_status()
        return response.json()

In [7]:

url_metadados = f"https://servicodados.ibge.gov.br/api/v3/agregados/{tabela_lavouras_temporárias}/metadados"

dados = requisicao_certificada(url_metadados)

print("Tabela:", dados["nome"])

for variavel in dados["variaveis"]:
    print(
        variavel["id"],
        variavel["nome"],
        variavel["unidade"]
    )

for classificacao in dados["classificacoes"]:
    print("Classificação:", classificacao["id"], classificacao["nome"])

    for categoria in classificacao["categorias"]:
        print(categoria["id"], categoria["nome"])

for classificacao in dados["classificacoes"]:
    print(classificacao["id"], classificacao["nome"])

Tabela: Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
109 Área plantada Hectares
1000109 Área plantada - percentual do total geral %
216 Área colhida Hectares
1000216 Área colhida - percentual do total geral %
214 Quantidade produzida Toneladas
112 Rendimento médio da produção Quilogramas por Hectare
215 Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
1000215 Valor da produção - percentual do total geral %
Classificação: 81 Produto das lavouras temporárias
0 Total
2688 Abacaxi*
83374 Abóbora
40471 Alfafa fenada
83375 Alface
2689 Algodão herbáceo (em caroço)
2690 Alho
2691 Amendoim (em casca)
2692 Arroz (em casca)
2693 Aveia (em grão)
2694 Batata-doce
2695 Batata-inglesa
2696 Cana-de-açúcar
40470 Cana para forragem
83384 Canola
2697 Cebola
83381 Cenoura
2698 Centeio (em grão)
2699 Cevada (em grão)
83379 

area plantada por regiao

In [8]:
periodo = "2015-2025"


url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{area_plant_100}"
)

parametros = {
    "localidades": "N2[all]",
    "classificacao": f"81[{id_soja},{id_cana},{id_milho}]"
}

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

linhas = []

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                linhas.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_regiao": serie["localidade"]["id"],
                    "regiao": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": valor
                })

df_area = pd.DataFrame(linhas)
df_area = df_area.sort_values(["regiao","cultura","ano","valor"])

df_area.head()

,codigo_cultura,cultura,codigo_regiao,regiao,ano,variavel,unidade,valor
99,2696,Cana-de-açúcar,5,Centro-Oeste,2015,Área plantada - percentual do total geral,Percentual,7.05
100,2696,Cana-de-açúcar,5,Centro-Oeste,2016,Área plantada - percentual do total geral,Percentual,7.22
101,2696,Cana-de-açúcar,5,Centro-Oeste,2017,Área plantada - percentual do total geral,Percentual,6.80
102,2696,Cana-de-açúcar,5,Centro-Oeste,2018,Área plantada - percentual do total geral,Percentual,6.98
103,2696,Cana-de-açúcar,5,Centro-Oeste,2019,Área plantada - percentual do total geral,Percentual,6.76


area plantada por estado

In [9]:
parametros["localidades"] = "N3[all]"

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                linhas.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_estado": serie["localidade"]["id"],
                    "estado": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": valor
                })

df_area = pd.DataFrame(linhas)
df_area = df_area.sort_values(["estado","cultura","ano","valor"])

df_area.head()

,codigo_cultura,cultura,codigo_regiao,regiao,ano,variavel,unidade,valor,codigo_estado,estado
473,2696,Cana-de-açúcar,NaN,NaN,2015,Área plantada - percentual do total geral,Percentual,3.46,12,Acre
474,2696,Cana-de-açúcar,NaN,NaN,2016,Área plantada - percentual do total geral,Percentual,3.26,12,Acre
475,2696,Cana-de-açúcar,NaN,NaN,2017,Área plantada - percentual do total geral,Percentual,2.80,12,Acre
476,2696,Cana-de-açúcar,NaN,NaN,2018,Área plantada - percentual do total geral,Percentual,0.71,12,Acre
477,2696,Cana-de-açúcar,NaN,NaN,2019,Área plantada - percentual do total geral,Percentual,0.74,12,Acre


In [10]:
parametros["localidades"] = "N6[all]"
periodo = 2015
url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{quant_prod_T}"
)

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                linhas.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_municipio": serie["localidade"]["id"],
                    "municipio": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": valor
                })

df_area = pd.DataFrame(linhas)
df_area = df_area.sort_values(["municipio","cultura","valor"])

df_area.head()

,codigo_cultura,cultura,codigo_regiao,regiao,ano,variavel,unidade,valor,codigo_estado,estado,codigo_municipio,municipio
11937,2696,Cana-de-açúcar,NaN,NaN,2015,Quantidade produzida,Toneladas,-,NaN,NaN,5200050,Abadia de Goiás - GO
17501,2711,Milho (em grão),NaN,NaN,2015,Quantidade produzida,Toneladas,1300,NaN,NaN,5200050,Abadia de Goiás - GO
6373,2713,Soja (em grão),NaN,NaN,2015,Quantidade produzida,Toneladas,1400,NaN,NaN,5200050,Abadia de Goiás - GO
8863,2696,Cana-de-açúcar,NaN,NaN,2015,Quantidade produzida,Toneladas,-,NaN,NaN,3100104,Abadia dos Dourados - MG
14427,2711,Milho (em grão),NaN,NaN,2015,Quantidade produzida,Toneladas,5587,NaN,NaN,3100104,Abadia dos Dourados - MG


In [ ]:
parametros["localidades"] = "N6[all]"
periodo = 2015
url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{area_plant_H}"
)

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                linhas.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_municipio": serie["localidade"]["id"],
                    "municipio": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": valor
                })

df_area = pd.DataFrame(linhas)
df_area = df_area.sort_values(["municipio","cultura","valor"])

df_area.head()